# CS781 Phase 2 – Legal NER – Ensemble of the finished runs (no training)

This notebook **trains nothing**. It finds the finished runs you attach as inputs, rebuilds each one from its saved weights,
and averages their per-token predictions:

| run | what must be attached as an input |
|---|---|
| Model 4 + LoRMA | its output folder (`phaseA_lorma_weights.pt`, `phaseB_lorma_weights.pt`, `results.json`) |
| Model 4 + LoRA | its output folder (`phaseA_lora_weights.pt`, `phaseB_lora_weights.pt`, `results.json`) |
| Model 1 + LoRA | its output folder (`phaseA_trainable_weights.pt`, `phaseB_trainable_weights.pt`, `results.json`) |
| KD student (Model 1) | its output folder (`logprobs_fold2_phaseA.npz`, `logprobs_fold3_phaseB.npz`) |
| the two Phase-1 backbones | `legal-gqa-final` (Model 1) and `legal-gqamoe-final` (Model 4) |
| the cleaned data | `ner_data_clean` |

A run that is not attached is simply left out. Each run's fold2 score is re-computed from its Phase-A weights and compared with the score
it reported (a mismatch is flagged), then the ensemble is scored on fold2 (all subsets), and the final test predictions use the
Phase-B (fold1+fold2) weights of the chosen members. Output: `submission.zip` with `prediction.json`.

In [ ]:
import os, sys, glob, json, time, math, random, shutil, itertools, contextlib
SESSION_START = time.time()
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ======================= SETTINGS (change here) =======================
EXPERIMENT_NAME = "ensemble_models_1_4_kd"

ENSEMBLE_MODE = "logprob"   # "logprob" = mean of log-probabilities | "prob" = mean of probabilities
EXCLUDE = []                # leave out runs whose folder path contains one of these texts, e.g. ["plus", "model1"]
FINAL_MEMBERS = "good"      # "good" = every run whose fold2 F1 is within MAX_GAP of the best single run (robust default)
                            # "all" | "best" = the best-scoring subset on fold2 (can over-fit: fold2 is small) | a list of member names
MAX_GAP = 0.04
CHECK_TOLERANCE = 0.01      # a rebuilt member whose fold2 score differs from its reported one by more than this is flagged
MAX_SUBSET_MEMBERS = 6      # above this many members only singles, leave-one-out and "all" are scored
DECODE = "viterbi"          # "viterbi" or "argmax"
VOTE = "auto"               # same-name voting for APP/RESP and A.COUNSEL/R.COUNSEL: "auto" = only if it improves the ensemble on fold2
BATCH_WINDOWS = 16          # windows per forward pass at inference (lower it if the GPU runs out of memory)

# window settings used only for a run that has no results.json (normally they are read from each run's results.json)
MAX_LEN = 1024
OVERLAP = 256
HEADER_TOKENS = 256
ADD_EOS_PREFIX = True

# --- paths (None = search INPUT_ROOT automatically) ---
INPUT_ROOT = "/kaggle/input"
DATA_DIR = None
M1_DIR = None               # Model 1 Phase-1 folder (legal-gqa-final)
M4_DIR = None               # Model 4 Phase-1 folder (legal-gqamoe-final)
TOKENIZER_DIR = None
OUT_DIR = "/kaggle/working/ner_ensemble"
SEED = 42

# kept only so the shared data/window cells run unchanged
SMOKE_TEST = False
SMOKE_DOCS = 3
BATCH_SIZE = 4
# [[END OF SETTINGS]]

for name in ["INPUT_ROOT", "DATA_DIR", "M1_DIR", "M4_DIR", "TOKENIZER_DIR", "OUT_DIR"]:
    env_value = os.environ.get("NER_" + name)
    if env_value:
        globals()[name] = env_value
os.makedirs(OUT_DIR, exist_ok=True)
DEFAULT_WINDOWS = {"MAX_LEN": MAX_LEN, "OVERLAP": OVERLAP, "HEADER_TOKENS": HEADER_TOKENS, "ADD_EOS_PREFIX": ADD_EOS_PREFIX}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE, "| torch", torch.__version__)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

try:
    import peft
except ImportError:
    os.system("pip install -q peft")
    import peft
from peft import LoraConfig, get_peft_model
print("peft", peft.__version__)

# Kaggle images can ship a torchao that is too old for the installed peft, which makes get_peft_model crash.
# We never use torchao, so tell peft it is not available.
try:
    peft.import_utils.is_torchao_available()
except ImportError as err:
    print("peft/torchao version clash ignored:", str(err)[:100])
    def _torchao_off():
        return False
    for _name, _mod in list(sys.modules.items()):
        if _name.startswith("peft") and hasattr(_mod, "is_torchao_available"):
            _mod.is_torchao_available = _torchao_off

# hide a harmless transformers warning about the custom model class (the weights are verified when loading)
import logging
from transformers.utils import logging as hf_logging
class _DropClassWarning(logging.Filter):
    def filter(self, record):
        return "to instantiate a model of type" not in record.getMessage()
for _h in hf_logging._get_library_root_logger().handlers:
    _h.addFilter(_DropClassWarning())


## 1. Model code (copied unchanged from the Phase-1 notebooks: Model 1 and Model 4)

In [ ]:
%%writefile /kaggle/working/legal_gqa_model.py
"""
Model 1 for CS781 Phase-1: GQA + RoPE + SwiGLU + RMSNorm decoder-only transformer.

Wrapped as a transformers PretrainedConfig/PreTrainedModel subclass so it is a
drop-in replacement for GPT2Config/GPT2LMHeadModel in the starter notebook:
same save_pretrained/from_pretrained, same Trainer + TrainingArguments +
resume_from_checkpoint behavior, same model.generate() call signature.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast


class LegalGQAConfig(PretrainedConfig):
    model_type = "legal_gqa"

    def __init__(
        self,
        vocab_size=50257,
        n_positions=1024,
        n_embd=1024,
        n_layer=8,
        n_head=16,
        n_kv_head=4,
        d_ff=2688,
        rope_theta=10000.0,
        dropout=0.1,
        initializer_range=0.02,
        rms_norm_eps=1e-5,
        bos_token_id=50256,
        eos_token_id=50256,
        pad_token_id=50256,
        activation_checkpointing=False,
        tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.vocab_size = vocab_size
        self.n_positions = n_positions
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.d_ff = d_ff
        self.rope_theta = rope_theta
        self.dropout = dropout
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = True
        # own implementation (HF's gradient_checkpointing flag is not supported here)
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    # q, k: (B, n_head, T, head_dim); cos/sin: (T, head_dim)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


class GroupedQueryAttention(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.n_rep = self.n_head // self.n_kv_head
        self.dropout_p = config.dropout

        self.q_proj = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_head * self.head_dim, config.n_embd, bias=False)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)
        # RoPE tables are fp32, v comes out of a linear (fp16 under autocast): force one dtype
        q, k = q.to(v.dtype), k.to(v.dtype)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        present = (k, v) if use_cache else None

        # GQA: broadcast each KV head across its group of query heads
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_bias,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        return self.o_proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.up_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.down_proj = nn.Linear(config.d_ff, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.ffn = SwiGLU(config)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        attn_out, present = self.attn(
            self.ln_1(x), cos, sin, attn_bias, is_causal, past_key_value, use_cache
        )
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, present


class LegalGQAForCausalLM(PreTrainedModel):
    config_class = LegalGQAConfig
    # input embedding and output head share one weight; must be declared or
    # save_pretrained / Trainer checkpointing refuses to save
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        # scaled init on residual-stream output projections (GPT-2 recipe)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(
                    p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer)
                )

    def get_input_embeddings(self):
        return self.wte

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def set_input_embeddings(self, value):
        self.wte = value

    def _rope_slice(self, past_len, T, device, dtype):
        # Computed on the fly (float32, then cast). Deliberately NOT stored as a buffer:
        # non-persistent buffers come back uninitialized after from_pretrained().
        inv_freq = 1.0 / (
            self.config.rope_theta
            ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim)
        )
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        """Returns (attn_bias, is_causal). attn_bias is None when the fast
        SDPA is_causal fused-kernel path is usable (no padding to mask)."""
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1

        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min

        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None, :, :]
        return bias, False

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        use_cache=None,
    ):
        # NOTE: no **kwargs here on purpose. If forward() accepts **kwargs, Trainer assumes the
        # model normalises its own loss, then multiplies it by n_gpu and skips the
        # gradient-accumulation division (loss reported 2x-64x too high).
        #
        # fp16 autocast is switched on HERE rather than relying on Trainer: under nn.DataParallel
        # on Kaggle the matmuls were measurably running in fp32 (no tensor cores, ~4x slower).
        # Only used for unpadded batches (all of training/eval); padded generation keeps the
        # exact fp32 path, since fp16 additive masks can overflow to -inf.
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False  # never keep K/V during training
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(
            attention_mask, past_len, T, input_ids.device, x.dtype
        )

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                x = torch_checkpoint(
                    lambda h, blk=block: blk(h, cos, sin, attn_bias, is_causal, None, False)[0],
                    x,
                    use_reentrant=False,
                )
            else:
                x, present = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        # when labels are given only the loss is needed; returning the full
        # (B, T, vocab) logits would just be gathered across GPUs for nothing
        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

    # No prepare_inputs_for_generation / GenerationMixin here on purpose —
    # generation is done by generate_utils.generate_batch(), which calls
    # this model's forward() directly with its own tuple-based KV cache
    # rather than going through model.generate().

In [ ]:
%%writefile /kaggle/working/moe.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class MoE(nn.Module):
    def __init__(self, d_model, n_routed, n_shared, top_k, expert_d_ff, aux_loss_coef=0.01):
        super().__init__()
        self.n_routed, self.top_k = n_routed, top_k
        self.aux_loss_coef = aux_loss_coef
        self.router = nn.Linear(d_model, n_routed, bias=False)
        self.routed_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_routed)])
        self.shared_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_shared)])
        self.last_aux_loss = None

    def forward(self, x):
        B, T, D = x.shape
        flat = x.reshape(-1, D)                                 # (N, D), N = B*T
        N = flat.shape[0]

        logits = self.router(flat)                               # (N, n_routed)
        probs = F.softmax(logits, dim=-1)
        top_w, top_i = probs.topk(self.top_k, dim=-1)             # (N, top_k) each
        top_w = top_w / top_w.sum(dim=-1, keepdim=True)           # renormalize among selected

        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.routed_experts):
            token_slot, k_slot = (top_i == e).nonzero(as_tuple=True)   # which tokens picked expert e, and at which of their top_k slots
            if token_slot.numel() == 0:
                continue
            expert_out = expert(flat[token_slot])                       # only THIS expert's assigned tokens, real compute saving
            out.index_add_(0, token_slot, expert_out * top_w[token_slot, k_slot].unsqueeze(-1))

        for expert in self.shared_experts:
            out = out + expert(flat)                                    # always on, every token

        if self.training:
            # Switch-Transformer style load-balancing aux loss: encourages the router
            # to spread tokens evenly across experts rather than collapsing onto a few.
            with torch.no_grad():
                assign_count = torch.zeros(self.n_routed, device=x.device)
                assign_count.scatter_add_(0, top_i.reshape(-1), torch.ones_like(top_i.reshape(-1), dtype=torch.float))
                frac_tokens = assign_count / (N * self.top_k)
            frac_prob = probs.mean(dim=0)
            self.last_aux_loss = self.aux_loss_coef * self.n_routed * (frac_tokens * frac_prob).sum()
        else:
            self.last_aux_loss = None

        return out.view(B, T, D)

In [ ]:
%%writefile /kaggle/working/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

from legal_gqa_model import RMSNorm, GroupedQueryAttention, LegalGQAConfig
from moe import MoE


class LegalGQAMoEConfig(PretrainedConfig):
    model_type = "legal_gqa_mha_moe"

    def __init__(
        self,
        vocab_size=50257, n_positions=1024, n_embd=1024, n_layer=9,
        mha_layers=(0, 4, 8),           # 0-indexed: layers 1, 5, 9 -- full MHA (n_kv_head = n_head)
        n_head=16, gqa_n_kv_head=4,     # the other 6 layers: GQA, same n_kv_head model 1/3 already proved
        n_routed_experts=6, n_shared_experts=1, moe_top_k=2, expert_d_ff=896, moe_aux_loss_coef=0.01,
        rope_theta=10000.0,
        dropout=0.0, initializer_range=0.02, rms_norm_eps=1e-5,
        bos_token_id=50256, eos_token_id=50256, pad_token_id=50256,
        activation_checkpointing=False, tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0
        self.vocab_size, self.n_positions, self.n_embd, self.n_layer = vocab_size, n_positions, n_embd, n_layer
        self.mha_layers = list(mha_layers)
        self.n_head, self.gqa_n_kv_head = n_head, gqa_n_kv_head
        self.n_routed_experts, self.n_shared_experts = n_routed_experts, n_shared_experts
        self.moe_top_k, self.expert_d_ff, self.moe_aux_loss_coef = moe_top_k, expert_d_ff, moe_aux_loss_coef
        self.rope_theta = rope_theta
        self.dropout, self.initializer_range, self.rms_norm_eps = dropout, initializer_range, rms_norm_eps
        self.use_cache = True
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id, eos_token_id=eos_token_id, pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings, **kwargs,
        )


class MixedDecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAMoEConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        self.is_mha = layer_idx in config.mha_layers
        n_kv_head = config.n_head if self.is_mha else config.gqa_n_kv_head
        # GroupedQueryAttention reads n_head/n_kv_head/n_embd/dropout off a config
        # object -- reusing it unmodified means building one small per-layer config
        # with just n_kv_head varied, rather than touching model 1's tested file at all
        attn_config = LegalGQAConfig(
            vocab_size=config.vocab_size, n_positions=config.n_positions,
            n_embd=config.n_embd, n_head=config.n_head, n_kv_head=n_kv_head,
            dropout=config.dropout,
        )
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(attn_config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.moe = MoE(config.n_embd, config.n_routed_experts, config.n_shared_experts,
                       config.moe_top_k, config.expert_d_ff, config.moe_aux_loss_coef)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_kv=None, use_cache=False):
        attn_out, present = self.attn(self.ln_1(x), cos, sin, attn_bias, is_causal, past_kv, use_cache)
        x = x + attn_out
        moe_out = self.moe(self.ln_2(x))
        x = x + moe_out
        return x, present, self.moe.last_aux_loss


class LegalGQAMoEForCausalLM(PreTrainedModel):
    config_class = LegalGQAMoEConfig
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAMoEConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([MixedDecoderBlock(config, i) for i in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer))

    def get_input_embeddings(self):
        return self.wte

    def set_input_embeddings(self, value):
        self.wte = value

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def _rope_slice(self, past_len, T, device, dtype):
        inv_freq = 1.0 / (self.config.rope_theta ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim))
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1
        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min
        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None]
        return bias, False

    def forward(self, input_ids, attention_mask=None, past_key_values=None, labels=None, use_cache=None):
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)
        # no **kwargs above on purpose -- Trainer treats a forward() accepting **kwargs
        # as self-normalizing its loss, breaking grad-accum scaling (models 1/2/3's note)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(attention_mask, past_len, T, input_ids.device, x.dtype)

        presents = [] if use_cache else None
        moe_losses = []
        for block in self.blocks:
            pkv = past_key_values[block.layer_idx] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                def run(h, blk=block):
                    h2, _, moe_l = blk(h, cos, sin, attn_bias, is_causal, None, False)
                    if moe_l is not None:
                        moe_losses.append(moe_l)
                    return h2
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x, present, moe_l = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)
                if moe_l is not None:
                    moe_losses.append(moe_l)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100,
            )
            if moe_losses:
                loss = loss + torch.stack(moe_losses).mean()

        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

## 2. Find data, backbones and tokenizer

In [ ]:
def find_files(filename):
    return sorted(glob.glob(os.path.join(INPUT_ROOT, "**", filename), recursive=True))

def pick_data_file(filename):
    if DATA_DIR is not None:
        found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    else:
        found = find_files(filename)
    if not found:
        raise FileNotFoundError(filename + " not found. Attach the cleaned data (ner_data_clean) as an input.")
    # prefer the cleaned copy (output of the data-prep notebook) over any raw copy
    found = sorted(found, key=lambda p: ("ner_data_clean" not in p, p))
    return found[0]

FOLD1_TEXT = pick_data_file("fold1-text.json")
FOLD1_GOLD = pick_data_file("fold1-gold.json")
FOLD2_TEXT = pick_data_file("fold2-text.json")
FOLD2_GOLD = pick_data_file("fold2-gold.json")
FOLD3_TEXT = pick_data_file("fold3-text.json")
LABELS_TXT = pick_data_file("labels.txt")
print("fold1:", FOLD1_TEXT)
print("fold3:", FOLD3_TEXT)

def find_backbone(model_type, forced):
    if forced is not None:
        return forced
    found = []
    for cfg_path in find_files("config.json"):
        folder = os.path.dirname(cfg_path)
        try:
            mt = json.load(open(cfg_path)).get("model_type")
        except Exception:
            continue
        has_weights = os.path.exists(os.path.join(folder, "model.safetensors")) or \
                      os.path.exists(os.path.join(folder, "pytorch_model.bin"))
        if mt == model_type and has_weights:
            found.append(folder)
    if not found:
        return None
    final_ones = [d for d in found if "checkpoint-" not in d]
    if final_ones:
        return final_ones[0]
    return max(found, key=lambda d: int(d.rsplit("-", 1)[1]))

M1_PATH = find_backbone("legal_gqa", M1_DIR)
M4_PATH = find_backbone("legal_gqa_mha_moe", M4_DIR)
print("Model 1 backbone:", M1_PATH)
print("Model 4 backbone:", M4_PATH)

def find_tokenizer_dir():
    if TOKENIZER_DIR is not None:
        return TOKENIZER_DIR
    for p in (M1_PATH, M4_PATH):
        if p is not None and os.path.exists(os.path.join(p, "tokenizer.json")):
            return p
    found = [os.path.dirname(p) for p in find_files("tokenizer.json") if "checkpoint-" not in p]
    if not found:
        raise FileNotFoundError("No tokenizer.json found. Attach your tokenizer dataset.")
    return found[0]

TOK_PATH = find_tokenizer_dir()
print("Tokenizer:", TOK_PATH)


## 3. Load data

In [ ]:
def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def load_texts(path):
    raw = read_json(path)
    texts = {}
    if isinstance(raw, dict):
        for key, value in raw.items():
            if isinstance(value, dict):
                value = value.get("text", "")
            texts[str(key)] = value
    else:
        for item in raw:
            key = item.get("id", item.get("doc_id"))
            texts[str(key)] = item["text"]
    return texts

def first_key(d, names):
    for n in names:
        if n in d:
            return d[n]
    raise KeyError("none of " + str(names) + " in " + str(list(d.keys())))

def parse_annotation(a):
    # returns (start, end, label, text_or_None) exactly as stored in the file
    if isinstance(a, dict):
        s = first_key(a, ["start", "start_char", "start_offset", "begin"])
        e = first_key(a, ["end", "end_char", "end_offset"])
        lab = first_key(a, ["label", "type", "entity_label"])
        txt = a.get("text", a.get("entity_text", None))
        return int(s), int(e), lab, txt
    txt = a[3] if len(a) > 3 else None
    return int(a[0]), int(a[1]), a[2], txt

def load_gold_raw(path):
    raw = read_json(path)
    gold = {}
    if isinstance(raw, dict):
        items = raw.items()
    else:
        items = [(it.get("id", it.get("doc_id")), it) for it in raw]
    for key, value in items:
        if isinstance(value, dict):
            value = value.get("entities", value.get("annotations", value.get("spans", [])))
        gold[str(key)] = [parse_annotation(a) for a in value]
    return gold

texts1 = load_texts(FOLD1_TEXT)
texts2 = load_texts(FOLD2_TEXT)
texts3 = load_texts(FOLD3_TEXT)
gold1_raw = load_gold_raw(FOLD1_GOLD)
gold2_raw = load_gold_raw(FOLD2_GOLD)
print("docs: fold1", len(texts1), "| fold2", len(texts2), "| fold3 (test)", len(texts3))

# ---- labels ----
DEFAULT_LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
LABELS = []
for line in open(LABELS_TXT, encoding="utf-8").read().splitlines():
    line = line.strip()
    if line:
        LABELS.append(line.replace(",", " ").split()[0])
if len(LABELS) != 12 or set(LABELS) != set(DEFAULT_LABELS):
    print("WARNING: labels.txt gave", LABELS, "-> using the 12 official labels instead")
    LABELS = DEFAULT_LABELS
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
N_TAGS = 1 + 2 * len(LABELS)      # O, then B-x / I-x for each label
TAG_NAMES = ["O"]
for l in LABELS:
    TAG_NAMES.append("B-" + l)
    TAG_NAMES.append("I-" + l)
print("labels:", LABELS)

# ---- end offset convention ----
exclusive_hits = 0
inclusive_hits = 0
with_text = 0
for texts, gold in [(texts1, gold1_raw), (texts2, gold2_raw)]:
    for doc_id, anns in gold.items():
        if doc_id not in texts:
            continue
        for s, e, lab, ent_text in anns:
            if ent_text is None:
                continue
            with_text += 1
            if texts[doc_id][s:e] == ent_text:
                exclusive_hits += 1
            if texts[doc_id][s:e + 1] == ent_text:
                inclusive_hits += 1
print(f"entity text check: {with_text} annotations | text[s:e]==ent: {exclusive_hits} | text[s:e+1]==ent: {inclusive_hits}")
if with_text == 0:
    print("WARNING: gold has no entity text, assuming END IS EXCLUSIVE (python slicing)")
    END_INCLUSIVE = False
else:
    END_INCLUSIVE = inclusive_hits > exclusive_hits
print("gold end offset is", "INCLUSIVE" if END_INCLUSIVE else "EXCLUSIVE")

def normalise_gold(gold_raw):
    # internal format everywhere: (start, end_exclusive, label)
    out = {}
    for doc_id, anns in gold_raw.items():
        spans = []
        for s, e, lab, ent_text in anns:
            if lab not in LABEL2ID:
                continue
            spans.append((s, e + 1 if END_INCLUSIVE else e, lab))
        out[doc_id] = spans
    return out

gold1 = normalise_gold(gold1_raw)
gold2 = normalise_gold(gold2_raw)

label_counts = {l: 0 for l in LABELS}
for spans in list(gold1.values()) + list(gold2.values()):
    for s, e, lab in spans:
        label_counts[lab] += 1
print("entity counts (fold1+fold2):", label_counts)

## 4. Tokenise

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOK_PATH)
tokenizer.model_max_length = 10**9          # we cut windows ourselves
assert tokenizer.is_fast, "need a fast tokenizer for offset mapping"
print("vocab:", len(tokenizer), "| eos:", tokenizer.eos_token_id, "| pad:", tokenizer.pad_token_id)

EOS_ID = tokenizer.eos_token_id
if tokenizer.pad_token_id is not None:
    PAD_ID = tokenizer.pad_token_id
elif EOS_ID is not None:
    PAD_ID = EOS_ID
else:
    PAD_ID = 0

def encode_doc(text):
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = list(enc["input_ids"])
    starts = []
    ends = []
    real = []
    for s, e in enc["offset_mapping"]:
        # byte-level BPE offsets can contain the leading space: trim whitespace
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        starts.append(s)
        ends.append(e)
        real.append(e > s)
    if ADD_EOS_PREFIX and EOS_ID is not None:
        ids = [EOS_ID] + ids
        starts = [0] + starts
        ends = [0] + ends
        real = [False] + real
    doc = {}
    doc["ids"] = np.array(ids, dtype=np.int64)
    doc["starts"] = np.array(starts, dtype=np.int64)
    doc["ends"] = np.array(ends, dtype=np.int64)
    doc["real"] = np.array(real, dtype=bool)
    return doc

def make_tags(doc, spans):
    # spans: (start, end_exclusive, label). Longest span wins if two overlap.
    n = len(doc["ids"])
    tags = np.zeros(n, dtype=np.int64)
    taken = np.zeros(n, dtype=bool)
    ordered = sorted(spans, key=lambda x: -(x[1] - x[0]))
    for s, e, lab in ordered:
        first = int(np.searchsorted(doc["ends"], s, side="right"))   # first token ending after s
        last = int(np.searchsorted(doc["starts"], e, side="left"))    # tokens starting before e are < last
        idx = [i for i in range(first, last) if doc["real"][i] and not taken[i]]
        if not idx:
            continue
        li = LABEL2ID[lab]
        for k, i in enumerate(idx):
            tags[i] = 1 + 2 * li if k == 0 else 2 + 2 * li
            taken[i] = True
    tags[~doc["real"]] = -100
    return tags

def tags_to_spans(tags, starts, ends):
    # tags / starts / ends only for REAL tokens. An I-x without matching B-x/I-x before it starts a new span.
    spans = []
    cur = None
    cs = 0
    ce = 0
    for i in range(len(tags)):
        t = int(tags[i])
        if t == 0:
            if cur is not None:
                spans.append((int(cs), int(ce), LABELS[cur]))
                cur = None
            continue
        lab = (t - 1) // 2
        is_begin = (t - 1) % 2 == 0
        if is_begin or cur != lab:
            if cur is not None:
                spans.append((int(cs), int(ce), LABELS[cur]))
            cur = lab
            cs = starts[i]
        ce = ends[i]
    if cur is not None:
        spans.append((int(cs), int(ce), LABELS[cur]))
    return spans

def build_docs(texts, gold=None):
    docs = []
    for doc_id, text in texts.items():
        doc = encode_doc(text)
        doc["id"] = doc_id
        doc["n_chars"] = len(text)
        doc["text"] = text
        if gold is not None:
            doc["gold"] = gold.get(doc_id, [])
            doc["tags"] = make_tags(doc, doc["gold"])
        docs.append(doc)
    return docs

t0 = time.time()
docs1 = build_docs(texts1, gold1)
docs2 = build_docs(texts2, gold2)
docs3 = build_docs(texts3)
print(f"tokenised in {time.time() - t0:.0f}s | tokens: fold1 {sum(len(d['ids']) for d in docs1):,} | fold2 {sum(len(d['ids']) for d in docs2):,} | fold3 {sum(len(d['ids']) for d in docs3):,}")

# offsets must be non-decreasing, otherwise searchsorted alignment is wrong
for d in docs1[:50] + docs3[:50]:
    assert np.all(np.diff(d["starts"]) >= 0) and np.all(np.diff(d["ends"]) >= 0), "token offsets not monotonic"

## 5. Metric, decoding

In [ ]:
def strict_scores(gold, pred):
    # gold / pred: {doc_id: [(start, end, label), ...]}; exact start+end+label match
    per_label = {}
    for lab in LABELS:
        tp = 0
        n_pred = 0
        n_gold = 0
        for doc_id in gold:
            g = set(x for x in gold[doc_id] if x[2] == lab)
            p = set(x for x in pred.get(doc_id, []) if x[2] == lab)
            tp += len(g & p)
            n_pred += len(p)
            n_gold += len(g)
        precision = tp / n_pred if n_pred > 0 else 0.0
        recall = tp / n_gold if n_gold > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        per_label[lab] = {"p": precision, "r": recall, "f1": f1, "tp": tp, "pred": n_pred, "gold": n_gold}
    # a label counts in the macro average if it has gold entities or predictions (all 12 do on the real data)
    used = [l for l in LABELS if per_label[l]["gold"] > 0 or per_label[l]["pred"] > 0]
    if not used:
        used = LABELS
    macro = {}
    macro["f1"] = float(np.mean([per_label[l]["f1"] for l in used]))
    macro["p"] = float(np.mean([per_label[l]["p"] for l in used]))
    macro["r"] = float(np.mean([per_label[l]["r"] for l in used]))
    return macro, per_label

# ---- Viterbi with BIO rules ----
def build_transitions():
    NEG = -1e4
    trans = np.zeros((N_TAGS, N_TAGS), dtype=np.float32)     # trans[from, to]
    start = np.zeros(N_TAGS, dtype=np.float32)
    for to in range(N_TAGS):
        if to > 0 and (to - 1) % 2 == 1:                     # to is I-x
            lab = (to - 1) // 2
            allowed = [1 + 2 * lab, 2 + 2 * lab]             # only after B-x or I-x
            for frm in range(N_TAGS):
                if frm not in allowed:
                    trans[frm, to] = NEG
            start[to] = NEG
    return trans, start

TRANS, START = build_transitions()

def viterbi(lp):
    n = lp.shape[0]
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    score = lp[0] + START
    back = np.zeros((n, N_TAGS), dtype=np.int16)
    for t in range(1, n):
        cand = score[:, None] + TRANS
        back[t] = cand.argmax(axis=0)
        score = cand.max(axis=0) + lp[t]
    best = np.zeros(n, dtype=np.int64)
    best[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        best[t - 1] = back[t, best[t]]
    return best

def decode_doc(lp, doc, mode):
    idx = np.where(doc["real"])[0]
    sub = lp[idx]
    if mode == "viterbi":
        tags = viterbi(sub)
    else:
        tags = sub.argmax(axis=1)
    return tags_to_spans(tags, doc["starts"][idx], doc["ends"][idx])

# ---- oracle: gold -> BIO -> spans. Shows the best score this tokenisation can ever reach ----
def oracle_check(docs, name):
    gold = {}
    pred = {}
    for d in docs:
        idx = np.where(d["real"])[0]
        gold[d["id"]] = d["gold"]
        pred[d["id"]] = tags_to_spans(d["tags"][idx], d["starts"][idx], d["ends"][idx])
    macro, per_label = strict_scores(gold, pred)
    print(f"ORACLE {name}: macro-F1 {macro['f1']:.4f} (P {macro['p']:.4f}, R {macro['r']:.4f})")
    bad = [(l, per_label[l]["f1"]) for l in LABELS if per_label[l]["f1"] < 0.99 and per_label[l]["gold"] > 0]
    if bad:
        print("   labels below 0.99:", [(l, round(f, 3)) for l, f in bad])
    return macro["f1"]

o1 = oracle_check(docs1, "fold1")
o2 = oracle_check(docs2, "fold2")
if min(o1, o2) < 0.97:
    print("!! Oracle is low: many gold spans do not start/end on token boundaries. Check END_INCLUSIVE above and the "
          "offset printout below before spending GPU time.")
    for d in docs1[:3]:
        text = texts1[d["id"]]
        for s, e, lab in d["gold"][:3]:
            print(repr(text[max(0, s - 10):s]), "[[", repr(text[s:e]), "]]", lab)

## 6. Windows

In [ ]:
def make_windows(n):
    # returns list of (start, end, keep_from). Tokens [keep_from, end) are the ones this window "owns";
    # [start, keep_from) is only warm-up context for the causal model.
    # The first window holds MAX_LEN tokens. Every later window gets the document header (first HEADER_TOKENS
    # tokens) put in front of it, so its body is HEADER_TOKENS shorter and the total stays <= MAX_LEN.
    wins = []
    first_end = min(MAX_LEN, n)
    wins.append((0, first_end, 0))
    prev_end = first_end
    body = MAX_LEN - HEADER_TOKENS
    while prev_end < n:
        st = prev_end - OVERLAP
        en = min(st + body, n)
        wins.append((st, en, prev_end))
        prev_end = en
    return wins

def head_len(st):
    # how many header tokens are put in front of a window that starts at token st
    if st > 0 and HEADER_TOKENS > 0:
        return HEADER_TOKENS
    return 0

def train_windows(docs):
    items = []
    for di, d in enumerate(docs):
        n = len(d["ids"])
        if n == 0:
            continue
        for st, en, keep in make_windows(n):
            items.append((di, st, en, keep))
    return items

def has_entity(doc, st, en, keep):
    seg = doc["tags"][keep:en]
    return bool((seg > 0).any())

def make_batch(docs, items, with_labels):
    B = len(items)
    L = max(head_len(st) + en - st for (di, st, en, keep) in items)
    ids = np.full((B, L), PAD_ID, dtype=np.int64)
    labels = np.full((B, L), -100, dtype=np.int64)
    lengths = np.zeros(B, dtype=np.int64)
    soft = None
    smask = None
    if with_labels and any("soft" in docs[di] for (di, st, en, keep) in items):
        soft = np.zeros((B, L, N_TAGS), dtype=np.float16)       # teacher scores
        smask = np.zeros((B, L), dtype=bool)                    # positions where the teacher term is used
    for j, (di, st, en, keep) in enumerate(items):
        d = docs[di]
        h = head_len(st)
        l = en - st
        if h > 0:
            ids[j, :h] = d["ids"][:h]            # document header as context (never trained, never predicted)
        ids[j, h:h + l] = d["ids"][st:en]
        lengths[j] = h + l
        if with_labels:
            t = d["tags"][st:en].copy()
            t[: keep - st] = -100        # warm-up positions are not trained (they are not used at inference)
            labels[j, h:h + l] = t
            if soft is not None and "soft" in d:
                owned = d["real"][st:en].copy()
                owned[: keep - st] = False                       # same positions as the gold labels
                smask[j, h:h + l] = owned
                soft[j, h:h + l] = d["soft"][st:en]
    out = {}
    out["soft"] = torch.from_numpy(soft) if soft is not None else None
    out["smask"] = torch.from_numpy(smask) if smask is not None else None
    out["ids"] = torch.from_numpy(ids)
    out["labels"] = torch.from_numpy(labels)
    out["lengths"] = torch.from_numpy(lengths)
    return out

n_win = len(train_windows(docs1))
print("fold1 windows:", n_win, "| steps per epoch:", math.ceil(n_win / BATCH_SIZE))

# sizes of the FULL run (used for the time estimate printed at the end of a smoke test)
FULL = {}
FULL["steps_per_epoch"] = math.ceil(n_win / BATCH_SIZE)
FULL["steps_per_epoch_all"] = math.ceil((n_win + len(train_windows(docs2))) / BATCH_SIZE)
FULL["val_windows"] = len(train_windows(docs2))
FULL["test_windows"] = len(train_windows(docs3))

if SMOKE_TEST:
    docs1 = docs1[:SMOKE_DOCS]
    docs2 = docs2[:SMOKE_DOCS]
    docs3 = docs3[:SMOKE_DOCS]
    texts3 = {d["id"]: texts3[d["id"]] for d in docs3}
    print(f"SMOKE: using {len(docs1)} / {len(docs2)} / {len(docs3)} docs (train / val / test)")

## 7. Rebuilding a trained run (LoRA / LoRMA adapters + head on a frozen Phase-1 backbone)

In [ ]:
sys.path.insert(0, "/kaggle/working")
from legal_gqa_model import LegalGQAForCausalLM
from model import LegalGQAMoEForCausalLM

# ---------------------------------------------------------------- LoRMA layer (written by hand, PEFT has none)
class LoRMALinear(nn.Module):
    """Frozen linear layer W0 (d_out x d_in) with a low-rank MULTIPLICATIVE update on its output side:
         plus:  W = ((alpha/r) B A + I) W0     B: d_out x r (zeros at start), A: r x d_out (random)
         perm:  W = P(B A) W0                  P rotates row i of B A cyclically by i; starts at P(BA) = I
       so training starts exactly at the Phase-1 weights."""
    def __init__(self, base, r, alpha, dropout, variant):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d = base.out_features
        self.d, self.r, self.variant = d, r, variant
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lorma_A = nn.Parameter(torch.zeros(r, d))
        self.lorma_B = nn.Parameter(torch.zeros(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))     # B stays 0 -> (scale*BA + I) = I
        else:
            with torch.no_grad():
                self.lorma_B.normal_(0.0, 0.02)
                self.lorma_B[:, 0] = 1.0                                # first column of B = ones, rest random
                self.lorma_A.zero_()
                self.lorma_A[0, 0] = 1.0                                # A[0,0] = 1, rest 0  ->  BA has ones in column 0 only
            ii = torch.arange(d).unsqueeze(1)
            jj = torch.arange(d).unsqueeze(0)
            self.register_buffer("rot_idx", (jj - ii) % d, persistent=False)   # rotated[i, j] = M[i, (j - i) mod d]

    def forward(self, x):
        if self.variant == "plus":
            z = self.base(x)                                            # W0 x
            return z + self.scale * F.linear(F.linear(self.drop(z), self.lorma_A), self.lorma_B)
        M = self.lorma_B @ self.lorma_A                                 # d x d, rank <= r
        R = M.gather(1, self.rot_idx)                                   # rank-inflated by the row rotation
        w = R.to(self.base.weight.dtype) @ self.base.weight             # P(BA) W0
        b = None if self.base.bias is None else R.to(self.base.bias.dtype) @ self.base.bias
        return F.linear(self.drop(x), w, b)

def add_lorma(root, targets, r, alpha, dropout, variant):
    names = []
    for name, module in list(root.named_modules()):
        for child_name, child in list(module.named_children()):
            if child_name in targets and isinstance(child, nn.Linear):
                setattr(module, child_name, LoRMALinear(child, r, alpha, dropout, variant))
                names.append(name + "." + child_name)
    return names



class NERModel(nn.Module):
    # same attribute names as the training notebooks (backbone / lstm / cls), so the saved weights load directly
    def __init__(self, backbone, n_classes, head):
        super().__init__()
        self.backbone = backbone
        d = backbone.config.n_embd
        self.head_type = head
        self.drop = nn.Dropout(0.1)
        if head == "bilstm":
            self.lstm = nn.LSTM(d, d // 2, batch_first=True, bidirectional=True)
        self.cls = nn.Linear(d, n_classes)

    def hidden(self, input_ids):
        b = self.backbone
        x = b.drop(b.wte(input_ids))
        cos, sin = b._rope_slice(0, input_ids.shape[1], input_ids.device, x.dtype)
        for blk in b.blocks:
            x = blk(x, cos, sin, None, True, None, False)[0]
        return b.ln_f(x)

    def forward(self, input_ids, lengths):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            h = self.hidden(input_ids)
            if self.head_type == "bilstm":
                T = h.shape[1]
                packed = nn.utils.rnn.pack_padded_sequence(h.float(), lengths.cpu(), batch_first=True, enforce_sorted=False)
                out, _ = self.lstm(packed)
                h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=T)
            logits = self.cls(h)
        return logits.float()

def build_member_model(m, state):
    path = M4_PATH if m["base"] == "m4" else M1_PATH
    if path is None:
        raise FileNotFoundError("backbone for %s not attached (%s)" % (m["name"], m["base"]))
    cls = LegalGQAMoEForCausalLM if m["base"] == "m4" else LegalGQAForCausalLM
    backbone = cls.from_pretrained(path)
    backbone.config.use_cache = False
    backbone.float()
    for p in backbone.parameters():
        p.requires_grad = False
    model = NERModel(backbone, N_TAGS, m["head"])
    if m["kind"] == "lorma":
        add_lorma(backbone, set(m["targets"]), m["r"], m["alpha"], 0.0, m["variant"])
    else:
        cfg = LoraConfig(r=m["r"], lora_alpha=m["alpha"], lora_dropout=0.0, target_modules=list(m["targets"]), bias="none")
        get_peft_model(backbone, cfg)
    params = dict(model.named_parameters())
    unknown = [k for k in state if k not in params]
    if unknown:
        raise RuntimeError("%s: %d saved tensors have no place in the rebuilt model, e.g. %s" % (m["name"], len(unknown), unknown[:3]))
    not_loaded = [n for n in params if n not in state and ("lora_" in n or "lorma_" in n or not n.startswith("backbone."))]
    if not_loaded:
        raise RuntimeError("%s: trained parameters missing from the saved file, e.g. %s" % (m["name"], not_loaded[:3]))
    with torch.no_grad():
        for n, v in state.items():
            if tuple(params[n].shape) != tuple(v.shape):
                raise RuntimeError("%s: shape mismatch for %s: saved %s vs rebuilt %s" % (m["name"], n, tuple(v.shape), tuple(params[n].shape)))
            params[n].copy_(v)
    model.to(DEVICE).eval()
    return model

@contextlib.contextmanager
def member_windows(w):
    names = ["MAX_LEN", "OVERLAP", "HEADER_TOKENS"]
    old = {n: globals()[n] for n in names}
    for n in names:
        globals()[n] = w[n]
    try:
        yield
    finally:
        for n in names:
            globals()[n] = old[n]


## 8. Prediction helpers, voting, submission writing

In [ ]:
@torch.no_grad()
def predict_logprobs(model, docs, batch_windows=16):
    # log-probabilities over the 25 tags for every token of every document
    model.eval()
    logprobs = [np.zeros((len(d["ids"]), N_TAGS), dtype=np.float32) for d in docs]
    items = []
    for di, d in enumerate(docs):
        if len(d["ids"]) == 0:
            continue
        for st, en, keep in make_windows(len(d["ids"])):
            items.append((di, st, en, keep))
    for i in range(0, len(items), batch_windows):
        chunk = items[i:i + batch_windows]
        batch = make_batch(docs, chunk, with_labels=False)
        logits = model(batch["ids"].to(DEVICE), batch["lengths"])
        lps = F.log_softmax(logits, dim=-1).cpu().numpy()
        for j, (di, st, en, keep) in enumerate(chunk):
            h = head_len(st)
            logprobs[di][keep:en] = lps[j, h + keep - st:h + en - st]
    return logprobs

def predict_docs(model, docs, batch_windows=16):
    # returns {doc_id: [(start, end_exclusive, label), ...]}
    logprobs = predict_logprobs(model, docs, batch_windows)
    preds = {}
    for di, d in enumerate(docs):
        preds[d["id"]] = decode_doc(logprobs[di], d, DECODE)
    return preds

def save_logprobs(model, docs, name):
    # per-token log-probs, kept so several models can be ensembled later
    lps = predict_logprobs(model, docs)
    tmp = os.path.join(OUT_DIR, name[:-4] + ".tmp.npz")
    np.savez_compressed(tmp, **{d["id"]: lp.astype(np.float16) for d, lp in zip(docs, lps)})
    os.replace(tmp, os.path.join(OUT_DIR, name))

def evaluate(model, docs):
    pred = predict_docs(model, docs)
    gold = {d["id"]: d["gold"] for d in docs}
    macro, per_label = strict_scores(gold, pred)
    return macro, per_label, pred

# ---- same-name voting inside one document ----
from collections import Counter
VOTE_GROUPS = [["APP", "RESP"], ["A.COUNSEL", "R.COUNSEL"]]     # labels that get confused with each other

def norm_surface(text):
    t = " ".join(text.lower().split())
    return t.strip(" .,;:()[]\"'")

def vote_spans(spans, text):
    # if the same name is predicted several times in a document with a mix of labels from one group,
    # give all of them the majority label (ties are left alone)
    out = list(spans)
    for group in VOTE_GROUPS:
        by_name = {}
        for i in range(len(out)):
            s, e, lab = out[i]
            if lab in group:
                key = norm_surface(text[s:e])
                if key not in by_name:
                    by_name[key] = []
                by_name[key].append(i)
        for key in by_name:
            idxs = by_name[key]
            if len(idxs) < 2:
                continue
            counts = Counter(out[i][2] for i in idxs).most_common()
            if len(counts) > 1 and counts[0][1] == counts[1][1]:
                continue
            winner = counts[0][0]
            for i in idxs:
                out[i] = (out[i][0], out[i][1], winner)
    return out

def postprocess(docs, pred):
    result = {}
    for d in docs:
        result[d["id"]] = vote_spans(pred.get(d["id"], []), d["text"])
    return result

def to_submission(pred):
    out = {}
    for doc_id, spans in pred.items():
        rows = []
        for s, e, lab in sorted(spans):
            end_out = e - 1 if END_INCLUSIVE else e
            rows.append([int(s), int(end_out), lab])
        out[doc_id] = rows
    return out

def check_submission(sub, texts):
    assert set(sub.keys()) == set(texts.keys()), "doc ids differ from the test file"
    n_spans = 0
    for doc_id, rows in sub.items():
        for s, e, lab in rows:
            end_excl = e + 1 if END_INCLUSIVE else e
            assert 0 <= s < end_excl <= len(texts[doc_id]), (doc_id, s, e)
            assert lab in LABEL2ID
            n_spans += 1
    print(f"submission OK: {len(sub)} docs, {n_spans} spans")

def save_submission(sub, name):
    path = os.path.join(OUT_DIR, name)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(sub, f)
    os.replace(tmp, path)
    print("wrote", path)

def show_samples(sub, texts, k=5):
    shown = 0
    for doc_id, rows in sub.items():
        for s, e, lab in rows[:2]:
            end_excl = e + 1 if END_INCLUSIVE else e
            print(f"  {doc_id} {lab:<10} [{s}:{e}] {texts[doc_id][s:end_excl]!r}")
            shown += 1
        if shown >= k:
            break

## 9. Find the finished runs

In [ ]:
WEIGHT_FILES = [("phaseA_lorma_weights.pt", "phaseB_lorma_weights.pt", "lorma"),
                ("phaseA_lora_weights.pt", "phaseB_lora_weights.pt", "lora"),
                ("phaseA_trainable_weights.pt", "phaseB_trainable_weights.pt", "lora")]

def read_json_safe(path):
    try:
        return json.load(open(path))
    except Exception:
        return None

def run_is_smoke(folder, res):
    if "_smoke" in folder:
        return True
    return bool(res and res.get("settings", {}).get("SMOKE_TEST", False))

def recorded_fold2(folder, res):
    pa = read_json_safe(os.path.join(folder, "phaseA_result.json"))
    if pa and "val_macro_f1" in pa:
        return float(pa["val_macro_f1"])
    if res and "phaseA" in res and res["phaseA"] and "val_macro_f1" in res["phaseA"]:
        return float(res["phaseA"]["val_macro_f1"])
    return None

def discover_members():
    members = []
    seen_folders = set()
    for fileA, fileB, kind in WEIGHT_FILES:
        for pa in find_files(fileA):
            folder = os.path.dirname(pa)
            if folder in seen_folders:
                continue
            res = read_json_safe(os.path.join(folder, "results.json"))
            if run_is_smoke(folder, res) or any(x in folder for x in EXCLUDE):
                continue
            seen_folders.add(folder)
            state = torch.load(pa, map_location="cpu")
            keys = list(state.keys())
            s = (res or {}).get("settings", None)
            exp = (res or {}).get("experiment", "")
            base = "m4" if any("expert" in k for k in keys) else ("m4" if "model4" in exp else "m1")
            head = "bilstm" if any(k.startswith("lstm.") for k in keys) else "linear"
            w = dict(DEFAULT_WINDOWS)
            if s is not None:                      # a run with a results.json used exactly those settings (older runs had no header context)
                w["MAX_LEN"] = s.get("MAX_LEN", 1024)
                w["OVERLAP"] = s.get("OVERLAP", 256)
                w["HEADER_TOKENS"] = s.get("HEADER_TOKENS", 0)
                w["ADD_EOS_PREFIX"] = s.get("ADD_EOS_PREFIX", True)
                head = s.get("HEAD", head)
            m = {"kind": kind, "base": base, "head": head, "folder": folder, "weightsA": pa, "windows": w,
                 "weightsB": os.path.join(folder, fileB) if os.path.exists(os.path.join(folder, fileB)) else None,
                 "recorded_f1": recorded_fold2(folder, res), "experiment": exp}
            if kind == "lorma":
                a_keys = [k for k in keys if k.endswith("lorma_A")]
                m["r"] = int(state[a_keys[0]].shape[0])
                m["targets"] = sorted({k.split(".")[-2] for k in a_keys})
                m["variant"] = (s or {}).get("LORMA_VARIANT", "plus")
                m["alpha"] = (s or {}).get("LORMA_ALPHA", m["r"])
                if s is None or "LORMA_VARIANT" not in s:
                    print("WARNING: no LORMA_VARIANT in results.json next to", pa, "-> assuming 'plus'. If the fold2 check below fails, try 'perm'.")
                m["name"] = "%s_lorma_%s" % (base, m["variant"])
            else:
                a_keys = [k for k in keys if k.endswith("lora_A.default.weight")]
                m["r"] = int(state[a_keys[0]].shape[0])
                m["targets"] = sorted({k.split(".")[-4] for k in a_keys})
                m["alpha"] = (s or {}).get("LORA_ALPHA", 32)
                m["variant"] = ""
                m["name"] = "%s_lora" % base
            del state
            members.append(m)
    # knowledge-distillation student: only its saved per-token scores are needed
    for p2 in find_files("logprobs_fold2_phaseA.npz"):
        folder = os.path.dirname(p2)
        res = read_json_safe(os.path.join(folder, "results.json"))
        if run_is_smoke(folder, res) or any(x in folder for x in EXCLUDE) or folder in seen_folders:
            continue
        p3b = os.path.join(folder, "logprobs_fold3_phaseB.npz")
        p3a = os.path.join(folder, "logprobs_fold3_phaseA.npz")
        if not (os.path.exists(p3b) or os.path.exists(p3a)):
            continue
        seen_folders.add(folder)
        kd_cfg = (res or {}).get("kd", {})
        alpha = kd_cfg.get("alpha", None)
        nm = "kd_student" if (alpha is None or alpha > 0) else "kd_baseline_student"
        members.append({"kind": "npz", "name": nm, "folder": folder, "fold2": p2,
                        "fold3": p3b if os.path.exists(p3b) else p3a, "fold3_is_phaseB": os.path.exists(p3b),
                        "recorded_f1": recorded_fold2(folder, res), "experiment": (res or {}).get("experiment", "")})
    # the same run attached twice (e.g. as a dataset and as a notebook output) must not count twice
    unique = []
    sigs = set()
    for m in members:
        f = m["weightsA"] if m["kind"] != "npz" else m["fold2"]
        sig = (m["kind"], os.path.getsize(f), None if m["recorded_f1"] is None else round(m["recorded_f1"], 6))
        if sig in sigs:
            print("duplicate of an earlier run, ignored:", m["folder"])
            continue
        sigs.add(sig)
        unique.append(m)
    members = unique
    # unique names
    counts = {}
    for m in members:
        counts[m["name"]] = counts.get(m["name"], 0) + 1
    used = {}
    for m in members:
        if counts[m["name"]] > 1:
            used[m["name"]] = used.get(m["name"], 0) + 1
            m["name"] = "%s_%d" % (m["name"], used[m["name"]])
    return members

MEMBERS = discover_members()
print("found %d finished runs:" % len(MEMBERS))
for m in MEMBERS:
    print("  %-22s %-6s recorded fold2 F1 %s | %s" % (m["name"], m["kind"], ("%.4f" % m["recorded_f1"]) if m["recorded_f1"] is not None else "n/a", m["folder"]))
    if m["kind"] != "npz":
        print("     base %s | rank %d | targets %s | head %s | windows %s/%s/%s | phase-B weights: %s" % (
            m["base"], m["r"], m["targets"], m["head"], m["windows"]["MAX_LEN"], m["windows"]["OVERLAP"], m["windows"]["HEADER_TOKENS"], "yes" if m["weightsB"] else "NO (phase A weights will be used)"))
if not MEMBERS:
    raise RuntimeError("No finished run found under " + INPUT_ROOT + ". Attach the output folders of the earlier notebooks as inputs.")


## 10. Per-token scores of every run: fold2 (Phase-A weights, honest) and fold3 / test (Phase-B weights)

In [ ]:
def check_shapes(lps, docs, name):
    for lp, d in zip(lps, docs):
        if lp.shape != (len(d["ids"]), N_TAGS):
            raise RuntimeError("%s: score array for %s has shape %s, expected %s (different tokenisation or EOS prefix?)" % (name, d["id"], lp.shape, (len(d["ids"]), N_TAGS)))

def load_npz_scores(path, docs, name):
    z = np.load(path)
    lps = []
    for d in docs:
        if d["id"] not in z.files:
            raise RuntimeError("%s: no scores for document %s in %s" % (name, d["id"], os.path.basename(path)))
        lps.append(z[d["id"]].astype(np.float32))
    check_shapes(lps, docs, name)
    return lps

def model_scores(m, which, docs):
    wpath = m["weightsA"] if which == "A" else (m["weightsB"] or m["weightsA"])
    state = torch.load(wpath, map_location="cpu")
    model = build_member_model(m, state)
    del state
    t0 = time.time()
    with member_windows(m["windows"]):
        lps = predict_logprobs(model, docs, BATCH_WINDOWS)
    check_shapes(lps, docs, m["name"])
    print("    %s weights (%s): %d docs in %.0fs" % (which, os.path.basename(wpath), len(docs), time.time() - t0))
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return lps

def cache_file(m, fold):
    st = os.stat(m["weightsA"] if m["kind"] != "npz" else m["fold2"])
    return os.path.join(OUT_DIR, "lp_%s_%s_%d.npz" % (m["name"], fold, st.st_size))

def cached(m, fold, docs, make):
    path = cache_file(m, fold)
    if os.path.exists(path):
        try:
            return load_npz_scores(path, docs, m["name"])
        except Exception:
            pass
    lps = make()
    tmp = path[:-4] + ".tmp.npz"
    np.savez_compressed(tmp, **{d["id"]: lp.astype(np.float16) for d, lp in zip(docs, lps)})
    os.replace(tmp, path)
    return lps

gold2_map = {d["id"]: d["gold"] for d in docs2}
LP2 = {}
LP3 = {}
SINGLE = {}
OK_MEMBERS = []
for m in MEMBERS:
    print("member", m["name"])
    try:
        if m["kind"] == "npz":
            LP2[m["name"]] = load_npz_scores(m["fold2"], docs2, m["name"])
            LP3[m["name"]] = load_npz_scores(m["fold3"], docs3, m["name"])
            if not m["fold3_is_phaseB"]:
                print("    note: only the Phase-A (fold1-only) test scores were found for this run")
        else:
            w = m["windows"]
            if w["ADD_EOS_PREFIX"] != ADD_EOS_PREFIX:
                raise RuntimeError("this run used ADD_EOS_PREFIX=%s, token positions would not line up" % w["ADD_EOS_PREFIX"])
            LP2[m["name"]] = cached(m, "f2", docs2, lambda: model_scores(m, "A", docs2))
            LP3[m["name"]] = cached(m, "f3", docs3, lambda: model_scores(m, "B", docs3))
        pred = {d["id"]: decode_doc(lp, d, DECODE) for d, lp in zip(docs2, LP2[m["name"]])}
        macro, per_label = strict_scores(gold2_map, pred)
        SINGLE[m["name"]] = {"f1": macro["f1"], "per_label": per_label}
        rec = m["recorded_f1"]
        flag = ""
        if rec is not None:
            flag = "  (reported %.4f -> %s)" % (rec, "OK" if abs(rec - macro["f1"]) <= CHECK_TOLERANCE else "MISMATCH, check settings")
        print("    fold2 strict macro-F1 %.4f%s" % (macro["f1"], flag))
        OK_MEMBERS.append(m["name"])
    except Exception as err:
        print("    SKIPPED:", type(err).__name__, str(err)[:300])
        LP2.pop(m["name"], None); LP3.pop(m["name"], None)
if not OK_MEMBERS:
    raise RuntimeError("no run could be scored")


## 11. Ensemble: score every subset of runs on fold2

In [ ]:
def combine(lp_list):
    if len(lp_list) == 1:
        return lp_list[0]
    if ENSEMBLE_MODE == "prob":
        return np.log(np.mean([np.exp(x) for x in lp_list], axis=0) + 1e-9)
    return np.mean(lp_list, axis=0)

def ensemble_pred(names, LP, docs):
    return {d["id"]: decode_doc(combine([LP[n][i] for n in names]), d, DECODE) for i, d in enumerate(docs)}

def f1_of(names):
    macro, per_label = strict_scores(gold2_map, ensemble_pred(names, LP2, docs2))
    return macro["f1"], per_label

n = len(OK_MEMBERS)
subsets = []
if n <= MAX_SUBSET_MEMBERS:
    for k in range(1, n + 1):
        subsets += [list(c) for c in itertools.combinations(OK_MEMBERS, k)]
else:
    subsets += [[x] for x in OK_MEMBERS] + [[y for y in OK_MEMBERS if y != x] for x in OK_MEMBERS] + [list(OK_MEMBERS)]
SUBSET_F1 = {}
t0 = time.time()
for s in subsets:
    SUBSET_F1[tuple(s)] = f1_of(s)[0]
print("scored %d subsets in %.0fs  (ENSEMBLE_MODE=%s)\n" % (len(subsets), time.time() - t0, ENSEMBLE_MODE))
ranked = sorted(SUBSET_F1.items(), key=lambda kv: (-kv[1], -len(kv[0])))
print("%-8s %s" % ("fold2 F1", "members"))
for names, f in ranked[:15]:
    print("%-8.4f %s" % (f, " + ".join(names)))
print("   ...")
singles = sorted([(f, names) for names, f in SUBSET_F1.items() if len(names) == 1], reverse=True)
print("best single run: %.4f (%s)" % (singles[0][0], singles[0][1][0]))
ALL = tuple(OK_MEMBERS)
all_f1 = SUBSET_F1.get(ALL, f1_of(list(ALL))[0])
print("all %d runs together: %.4f" % (n, all_f1))


## 12. Choose the members, vote, write the submission

In [ ]:
best_names, best_f1 = ranked[0]
best_single = singles[0][0]
if FINAL_MEMBERS == "all":
    CHOSEN = list(OK_MEMBERS)
elif FINAL_MEMBERS == "best":
    CHOSEN = list(best_names)
elif FINAL_MEMBERS == "good":
    CHOSEN = [x for x in OK_MEMBERS if SINGLE[x]["f1"] >= best_single - MAX_GAP]
    dropped = [x for x in OK_MEMBERS if x not in CHOSEN]
    if dropped:
        print("left out (fold2 F1 more than %.2f below the best single run): %s" % (MAX_GAP, dropped))
elif isinstance(FINAL_MEMBERS, (list, tuple)):
    CHOSEN = [x for x in FINAL_MEMBERS if x in OK_MEMBERS]
    assert CHOSEN, "none of FINAL_MEMBERS was scored: " + str(OK_MEMBERS)
else:
    raise ValueError("FINAL_MEMBERS must be 'good', 'all', 'best' or a list of member names")
print("final members:", CHOSEN)

ens_f1, ens_per_label = f1_of(CHOSEN)
pred2 = ensemble_pred(CHOSEN, LP2, docs2)
pred2_vote = postprocess(docs2, pred2)
macro_vote, per_label_vote = strict_scores(gold2_map, pred2_vote)
print("FOLD2 ensemble strict macro-F1 %.4f | with same-name voting %.4f" % (ens_f1, macro_vote["f1"]))
if VOTE == "auto":
    USE_VOTE = macro_vote["f1"] > ens_f1
else:
    USE_VOTE = bool(VOTE)
print("voting will be", "USED" if USE_VOTE else "NOT used")

print("\n%-10s" % "label" + "".join("%12s" % n_[:11] for n_ in CHOSEN[:4]) + "%12s" % "ENSEMBLE")
for lab in LABELS:
    row = "%-10s" % lab
    for n_ in CHOSEN[:4]:
        row += "%12.3f" % SINGLE[n_]["per_label"][lab]["f1"]
    row += "%12.3f" % ens_per_label[lab]["f1"]
    print(row)

pred3 = ensemble_pred(CHOSEN, LP3, docs3)
if USE_VOTE:
    pred3 = postprocess(docs3, pred3)
sub = to_submission(pred3)
check_submission(sub, texts3)
show_samples(sub, texts3)
save_submission(sub, "submission.json")

import zipfile
with zipfile.ZipFile(os.path.join(OUT_DIR, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(OUT_DIR, "submission.json"), arcname="prediction.json")     # Codabench wants prediction.json inside submission.zip

results = {"experiment": EXPERIMENT_NAME, "ensemble_mode": ENSEMBLE_MODE, "members_found": [{k: v for k, v in m.items() if k in ("name", "kind", "folder", "recorded_f1")} for m in MEMBERS],
           "single_fold2": {k: v["f1"] for k, v in SINGLE.items()}, "subset_fold2": {" + ".join(k): v for k, v in ranked},
           "chosen": CHOSEN, "fold2_ensemble_f1": ens_f1, "fold2_ensemble_f1_voted": macro_vote["f1"], "use_vote": USE_VOTE,
           "per_label_fold2": {l: ens_per_label[l]["f1"] for l in LABELS}, "end_inclusive": END_INCLUSIVE}
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=1, default=str)
print("\nDONE. Upload", os.path.join(OUT_DIR, "submission.zip"), "to Codabench. Files:", sorted(os.listdir(OUT_DIR)))


**Upload `submission.zip` (it contains `prediction.json`) to Codabench.** Fold2 is small (about 35 documents), so differences below about 0.01 between subsets are noise; the default (`FINAL_MEMBERS = "auto"`) therefore only drops runs when that clearly helps.